# Snapshot eval statistics tables — CSV + LaTeX export, per comparability group

Companion to `sps_eval_catplot_cmp.ipynb`: it uses the **same main input** (keep the
`SNAPSHOT_GROUPS` / `ACTIVE_GROUPS` block in sync) and exports, **for each active group**:

1. **one statistics table per metric** — rows `(algorithm, variant)`, all eval settings side by
   side as columns: the numbers behind that notebook's stat glyphs;
2. **one EV charging success table** — how many charging sessions reached the target SoC, at the
   reward's tolerance and strictly, and how many the policy ignored;
3. **one BES management table** — how the battery is used, with both "discharges early and never
   comes back" flags.
4. **one joint summary table**: return, cost and energy statistics next to the EV session counts
   and the headline BES numbers, one row per `(algorithm, variant)`.

## Why groups

A snapshot's rule-based (RBC) eval replays the episodes of *that snapshot's own* trial config, so
an RBC result only transfers to snapshots sharing it. The `_eval_ch` snapshots re-evaluate the
same checkpoint on the **training** EV profiles (`ev_0…5.csv`) rather than the held-out
`ev_eval_0…3.csv` — so `v2` and `v2_eval_ch` are a different episode population and may not share
a table. `old` and `regulator` additionally score under different reward stacks, which moves
`total_reward` onto its own scale. One group per reward/eval variant keeps every column of a table
comparable; the audit cell checks that against the snapshots rather than trusting their names.

A group's RBC rows come from the single snapshot named in `rbc-ref`; the other members' rule-based
runs are audited against it instead of being pooled in.

## Export format

Per active group and table, `<group>_<date>_<time>_out_<i>.{csv,tex}` lands next to this notebook
— the CSV and the LaTeX body of one table share the same `i`:

- a **CSV** at full precision, and
- a **LaTeX table body** (`.tex`), rounded, with `\midrule` between algorithm groups — meant to
  be `\input{}` into a hand-written booktabs `tabular` skeleton (a commented example sits at the
  top of every file), so rounding and ± assembly live in code while the LaTeX wrapper stays a
  thin shell.

Every body **ends with `\bottomrule`**: do NOT add another one after the `\input{}`. LaTeX inserts
file-hook tokens at the end of an `\input{}`ed file which start a new table cell, so a following
`\midrule`/`\bottomrule` fails with *Misplaced `\noalign`*. Close the skeleton with
`\end{tabular}` directly.

Run the **Imports** cell (right after the main input) before anything else.

**Main input:** `SNAPSHOT_GROUPS`, `ACTIVE_GROUPS`, `RBC_STRATEGIES`, `EVAL_CFGS`, `SEEDS`,
`METRICS`, `TEX_VARIANT_LABELS` and the EV / BES constants in the next cell.

In [1]:
# --- Main input ---------------------------------------------------------------------
# Snapshots are organised into COMPARABILITY GROUPS. A group is the unit that gets
# swapped: name groups in ACTIVE_GROUPS and re-run the notebook to regenerate every
# figure and table for them. Each active group is drawn on its own, never merged.
#
# Keys per group:
#   label            free-text description; printed in the log, handy for LaTeX captions
#   members          the policy sources drawn on the algorithm slots (keys below)
#   rbc-ref          snapshot whose runs/eval_rbc_* supplies this group's RBC strips.
#                    Only snapshots that actually HOLD a runs/eval_rbc_* directory work
#                    — at the time of writing that is the six PPO snapshots; the SAC ones
#                    carry no rule-based run, and pointing a group at one drops every RBC
#                    strip from that group (loudly: the discovery cell warns per strategy).
#   rbc-strategies   (optional) strategies to draw; defaults to RBC_STRATEGIES
# Keys per member:
#   path             snapshot dir (absolute, or relative to the repo root)
#   algo             x-axis category (also used in the PDF filenames)
#   variant          hue / dodge label inside the algorithm's slot
#   eval-kind        (optional) "eval-ray" | "eval-sb" | "eval-rbc"; default "eval-ray"
#   runs             (optional) restrict to these runs/<name> directories; default: all
#
# A group MAY hold several variants per algorithm (they then dodge side by side, as an
# earlier revision of this notebook drew them). The grouping below is deliberately one
# group per reward/eval variant, so `total_reward` is comparable everywhere inside a
# figure; the audit cell further down warns whenever a group mixes reward stacks or
# eval datasets.

# The 12 sta_lin_battery_EV snapshots, for reference:
#   20260721_093047_sta_lin_battery_EV_ppo              base            PPO
#   20260721_093047_sta_lin_battery_EV_ppo_eval_ch      base            PPO, eval on training EVs
#   20260721_101901_sta_lin_battery_EV_ppo_v2           v2              PPO
#   20260721_101901_sta_lin_battery_EV_ppo_v2_eval_ch   v2              PPO, eval on training EVs
#   20260721_102122_sta_lin_battery_EV_mo_ppo_old       old rewards     PPO
#   20260721_102226_sta_lin_battery_EV_ppo              regulator       PPO   <- same dir name as base, different run
#   20260721_103047_sta_lin_battery_EV_sac              base            SAC
#   20260721_103047_sta_lin_battery_EV_sac_eval_ch      base            SAC, eval on training EVs
#   20260721_111901_sta_lin_battery_EV_sac_v2           v2              SAC
#   20260721_111901_sta_lin_battery_EV_sac_v2_eval_ch   v2              SAC, eval on training EVs
#   20260722_123922_sta_lin_battery_EV_mo_sac_old       old rewards     SAC
#   20260722_134026_sta_lin_battery_EV_sac              regulator       SAC

SNAPSHOT_GROUPS: dict[str, dict] = {
    "base": {
        "label": "EVChargingSessionReward (charge_progress_scale 0.5) — held-out EV profiles",
        "members": [
            {"path": "snapshots/20260721_093047_sta_lin_battery_EV_ppo", "algo": "PPO", "variant": "base"},
            {"path": "snapshots/20260721_103047_sta_lin_battery_EV_sac", "algo": "SAC", "variant": "base"},
        ],
        "rbc-ref": "snapshots/20260721_093047_sta_lin_battery_EV_ppo",
    },
    "v2": {
        "label": "EVChargingSessionReward (charge_progress_scale 10.0) — held-out EV profiles",
        "members": [
            {"path": "snapshots/20260721_101901_sta_lin_battery_EV_ppo_v2", "algo": "PPO", "variant": "v2"},
            {"path": "snapshots/20260721_111901_sta_lin_battery_EV_sac_v2", "algo": "SAC", "variant": "v2"},
        ],
        "rbc-ref": "snapshots/20260721_101901_sta_lin_battery_EV_ppo_v2",
    },
    "old": {
        "label": "EVChargingRewardV0 (diff_threshold 0.02, soc_diff_multiplier 5.0) — held-out EV profiles",
        "members": [
            {"path": "snapshots/20260721_102122_sta_lin_battery_EV_mo_ppo_old", "algo": "PPO", "variant": "old rewards"},
            {"path": "snapshots/20260722_123922_sta_lin_battery_EV_mo_sac_old", "algo": "SAC", "variant": "old rewards"},
        ],
        "rbc-ref": "snapshots/20260721_102122_sta_lin_battery_EV_mo_ppo_old",
    },
    "regulator": {
        "label": "v2 stack plus BESRegulatorReward + EVRegulatorReward (penalty -1.0 each) — held-out EV profiles",
        "members": [
            {"path": "snapshots/20260721_102226_sta_lin_battery_EV_ppo", "algo": "PPO", "variant": "regulator rewards"},
            {"path": "snapshots/20260722_134026_sta_lin_battery_EV_sac", "algo": "SAC", "variant": "regulator rewards"},
        ],
        "rbc-ref": "snapshots/20260721_102226_sta_lin_battery_EV_ppo",
    },
    "base_eval_ch": {
        "label": "base checkpoints re-evaluated on the TRAINING EV profiles (ev_0…ev_5)",
        "members": [
            {"path": "snapshots/20260721_093047_sta_lin_battery_EV_ppo_eval_ch", "algo": "PPO", "variant": "base"},
            {"path": "snapshots/20260721_103047_sta_lin_battery_EV_sac_eval_ch", "algo": "SAC", "variant": "base"},
        ],
        "rbc-ref": "snapshots/20260721_093047_sta_lin_battery_EV_ppo_eval_ch",
    },
    "v2_eval_ch": {
        "label": "v2 checkpoints re-evaluated on the TRAINING EV profiles (ev_0…ev_5)",
        "members": [
            {"path": "snapshots/20260721_101901_sta_lin_battery_EV_ppo_v2_eval_ch", "algo": "PPO", "variant": "v2"},
            {"path": "snapshots/20260721_111901_sta_lin_battery_EV_sac_v2_eval_ch", "algo": "SAC", "variant": "v2"},
        ],
        "rbc-ref": "snapshots/20260721_101901_sta_lin_battery_EV_ppo_v2_eval_ch",
    },
}

# Groups to draw: a tuple of SNAPSHOT_GROUPS keys, or None for every declared group.
# THIS is the swap point — every table below (per-metric statistics, EV success counts,
# BES summary) is exported per active group and for nothing else.
#
# "base_eval_ch", "v2_eval_ch"
ACTIVE_GROUPS: tuple[str, ...] | None = ("base", "v2")

# Rule-based baselines: every strategy listed becomes one row in the group's "RBC" block,
# read from the group's `rbc-ref` snapshot. Drop entries to trim the block; a group may
# override the list with its own "rbc-strategies" key.
RBC_STRATEGIES = (
    # "do_nothing",
    # "deficit_discharge",
    # "pv_surplus_charge",
    # "self_coverage",
    # "price_median",
    # "price_median_scaled_0.25",
    # "price_median_scaled_0.5",
    # "price_median_scaled_0.75",
    # "price_median_autarky",
)

# Which eval settings become the table columns. None = every setting discovered: the
# held-out `eval_<i>` configs of the GEN layout (numerically sorted), and "default" for
# a flat STA layout (CSVs directly under eval_results/*_eval/). The sta_lin_battery_EV
# snapshots are STA, so this resolves to ("default",).
EVAL_CFGS: tuple[str, ...] | None = None

# Optional display names for the settings, keyed by the discovered `eval_<i>` label.
# Only used in the console log and in the LaTeX column heads you write around the
# exported bodies — the CSV header and the filenames keep the raw `eval_<i>`. Empty
# here: the only setting is "default", which needs no display name.
EVAL_CFG_LABELS: dict[str, str] = {}

# Eval seeds to keep, applied to every source after discovery:
#   SEEDS = None       keep every discovered seed (default)
#   SEEDS = 42         keep only the eval run seeded 42
#   SEEDS = [42, 43]   keep those two seeds, pooled into each strip
# Matched against the seed every run recorded itself (sidecar JSON "seed" for
# eval-ray/eval-sb, args.json "resolved_seed" for eval-rbc), never the directory name.
SEEDS: int | list[int] | None = [42]

# One exported table per metric: rows (algorithm, variant), columns the eval settings.
METRICS = ("total_reward", "cum_E_kWh", "cum_price_EUR")

# Optional display names for the LaTeX exports' variant column, keyed by the raw variant
# label. Only affects the .tex bodies — the CSVs and the tables displayed in this
# notebook keep the raw keys. Values are inserted as RAW LaTeX (no escaping — math like
# $\times$ is allowed, escape %, & and _ yourself); unlisted variants keep their raw
# label, LaTeX-escaped. Shorter names keep the table inside the A4 text block without
# shrinking the font further.
TEX_VARIANT_LABELS: dict[str, str] = {
    "old rewards": "old rewards",
    "regulator rewards": "regulator rewards",
    "do_nothing": "do nothing",
    "deficit_discharge": "deficit discharge",
    "self_coverage": "self coverage",
    "price_median": "price median",
    "price_median_scaled_0.25": "price median $\\times$0.25",
    "price_median_scaled_0.5": "price median $\\times$0.5",
    "price_median_scaled_0.75": "price median $\\times$0.75",
    "price_median_autarky": "price median autarky",
}

# Decimals of the exported LaTeX bodies (the CSVs keep full precision).
TEX_DECIMALS = 2

# --- EV charging success ------------------------------------------------------------
# A charging session ends when the charger releases the EV; it counts as met when the
# SoC reached by then is at/above the target. EV_SOC_TOLERANCE mirrors
# EVChargingSessionReward's `disconnect_soc_tolerance` (0.05 in every trial YAML of
# these snapshots), so the "met@tol" column reproduces the verdict the policy was
# actually rewarded on; "met@strict" drops the tolerance.
EV_SOC_TOLERANCE = 0.05
# A session is "ignored" when the EV's SoC never moves EV_IGNORED_SOC_CHANGE or more away
# from its arrival SoC while plugged in (charging or V2G): the policy left the car
# essentially untouched. 0.05 = 5 % SoC.
EV_IGNORED_SOC_CHANGE = 0.05

# --- BES management -----------------------------------------------------------------
# Two independent readings of "the battery only discharges early in the day":
#   throughput  — at least EARLY_SHARE_FLAG of the episode's |battery power| happens
#                 before EARLY_HOURS_END, and the episode recharges less than
#                 RECHARGE_RATIO_FLAG of what it discharged;
#   SoC shape   — after its lowest point the SoC never climbs back by more than
#                 SOC_RECOVERY_TOLERANCE, i.e. a drawdown followed by a flat line.
# An episode whose total battery throughput stays below IDLE_THROUGHPUT_KWH is reported
# as "idle" and flagged by neither criterion (an untouched battery is not a drawdown).
EARLY_HOURS_END = 8.0
EARLY_SHARE_FLAG = 0.70
RECHARGE_RATIO_FLAG = 0.20
SOC_RECOVERY_TOLERANCE = 0.02
IDLE_THROUGHPUT_KWH = 0.10
# Battery power below this magnitude counts as idle when charge/discharge direction
# changes are counted, so numerical dust does not inflate the cycle count.
POWER_DEADBAND_KW = 0.05

## Imports

Kept in their own cell, and deliberately the only place that imports anything: the cells below do
filesystem discovery and read hundreds of trajectory files, so they are slow enough to invite an
interrupt. Interrupting a cell **while it is importing** leaves a half-initialised module behind
in `sys.modules`, and every later import of it in that same kernel then fails with something like

```
AttributeError: partially initialized module 'pandas' has no attribute '_pandas_parser_CAPI'
(most likely due to a circular import)
```

That state cannot be repaired from inside the kernel — **restart the kernel** and run this cell
first. Isolating the imports here keeps an interrupted discovery run from causing it.

In [2]:
# Every import used by this notebook. Run this cell first; if it fails with a
# "partially initialized module" AttributeError, restart the kernel (see above) — no
# code below can undo that state.
import hashlib
import json
import re
from datetime import datetime
from functools import lru_cache
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

In [3]:
EVAL_KINDS = ("eval-ray", "eval-sb", "eval-rbc")

# A held-out eval-config result directory ends in `_eval_<i>` (e.g. battery_lin_power_eval_1);
# training-config directories (e.g. battery_lin_power_3) do not match and are skipped.
_EVAL_CFG_DIR_PATTERN = re.compile(r"(?:^|_)eval_(\d+)$")


# Walk upwards from the cwd until the directory containing snapshots/ is found, so the
# notebook runs both from thesis_eval/ and from the repo root.
def _find_repo_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "snapshots").is_dir():
            return candidate
    raise FileNotFoundError("No 'snapshots/' directory found upwards of the cwd — run inside the repo.")


REPO_ROOT = _find_repo_root()


def _snapshot_dir(path: str) -> Path:
    """Snapshot dir of a source `path`, resolved against the repo root when relative."""
    candidate = Path(path)
    return candidate if candidate.is_absolute() else REPO_ROOT / candidate


# Maps a runs/eval_* directory to its eval kind — mirrors the run-id scheme
# tools/snapshot/submit_snapshot.py builds from --kind.
def _run_dir_eval_kind(run_dir: Path) -> str:
    if run_dir.name.startswith("eval_rbc_"):
        return "eval-rbc"
    if run_dir.name.startswith("eval_sb_"):
        return "eval-sb"
    return "eval-ray"


# `eval_<i>` for a held-out eval-config result dir name, None for anything else.
def _cfg_dir_setting(dir_name: str) -> str | None:
    match = _EVAL_CFG_DIR_PATTERN.search(dir_name)
    return f"eval_{match.group(1)}" if match else None


# Ray/SB3 eval CSVs carry a sidecar JSON (same stem) with the run's trial seed + algorithm.
def _sidecar_seed_and_algorithm(csv_path: Path) -> tuple[int | None, str | None]:
    sidecar = csv_path.with_suffix(".json")
    if not sidecar.is_file():
        return None, None
    try:
        payload = json.loads(sidecar.read_text())
    except (json.JSONDecodeError, OSError):
        return None, None
    return payload.get("seed"), payload.get("algorithm")


# Rule-based runs record resolved_seed in an args.json above episodes.csv (directly
# above the per-strategy dir in the flat layout, one level further up with per-config
# dirs) — walk up towards the run dir and take the first hit.
def _rbc_resolved_seed(csv_path: Path, run_dir: Path) -> int | None:
    for candidate_dir in csv_path.parents:
        args_path = candidate_dir / "args.json"
        if args_path.is_file():
            try:
                return json.loads(args_path.read_text()).get("resolved_seed")
            except (json.JSONDecodeError, OSError):
                return None
        if candidate_dir == run_dir:
            return None
    return None


# Discovers one record per eval CSV of `source` across ALL eval settings kept by
# EVAL_CFGS (None = no filter). Both layouts are globbed; every record is tagged with
# its setting ("default" for the flat STA layout, `eval_<i>` for held-out GEN config
# dirs), which is what the per-setting figures are drawn from. The record also keeps the
# CSV's own directory, which is where that eval's `trajectories/` sits — the EV and BES
# analyses further down read exactly the runs that ended up in the figures.
def _discover_source_records(source: dict) -> list[dict]:
    eval_kind = source.get("eval-kind", "eval-ray")
    if eval_kind not in EVAL_KINDS:
        raise ValueError(f"'eval-kind' must be one of {EVAL_KINDS}, got {eval_kind!r}")

    snapshot_dir = _snapshot_dir(source["path"])
    if not snapshot_dir.is_dir():
        raise FileNotFoundError(f"Snapshot directory not found: {snapshot_dir}")

    run_dirs = [
        run_dir for run_dir in sorted((snapshot_dir / "runs").glob("eval_*"))
        if _run_dir_eval_kind(run_dir) == eval_kind
    ]
    if wanted_runs := source.get("runs"):
        run_dirs = [run_dir for run_dir in run_dirs if run_dir.name in set(wanted_runs)]
    if not run_dirs:
        raise FileNotFoundError(f"No '{eval_kind}' run directories found under {snapshot_dir / 'runs'}")

    records = []
    for run_dir in run_dirs:
        if eval_kind == "eval-rbc":
            # flat: .../<strategy>/episodes.csv — nested: .../<strategy>/<cfg>_eval_<i>/episodes.csv
            candidates = [
                (csv_path, "default", csv_path.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/episodes.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name), csv_path.parent.parent.name)
                for csv_path in sorted(run_dir.glob("eval_results/*_rule_based/*/*/episodes.csv"))
            ]
            strategies = sorted({strategy for _, _, strategy in candidates})
            strategy_wanted = source.get("rbc-strategy")
            if strategy_wanted is None and len(strategies) > 1:
                raise ValueError(
                    f"Source '{source['algo']} — {source['variant']}': multiple rule-based strategies "
                    f"{strategies} under {run_dir.name} — set 'rbc-strategy' to pick one."
                )
            for csv_path, setting, strategy in candidates:
                if setting is None or (strategy_wanted is not None and strategy != strategy_wanted):
                    continue
                records.append({
                    "group": source["group"], "algo": source["algo"], "variant": source["variant"],
                    "snapshot_dir": snapshot_dir, "run": run_dir.name, "setting": setting,
                    "seed": _rbc_resolved_seed(csv_path, run_dir), "eval_algorithm": strategy,
                    "csv": csv_path, "result_dir": csv_path.parent,
                })
        else:
            candidates = [
                (csv_path, "default")
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/eval_*.csv"))
            ] + [
                (csv_path, _cfg_dir_setting(csv_path.parent.name))
                for csv_path in sorted(run_dir.glob("eval_results/*_eval/*/eval_*.csv"))
            ]
            for csv_path, setting in candidates:
                if setting is None:
                    continue  # training-config result dir — not a held-out eval config
                seed, algorithm = _sidecar_seed_and_algorithm(csv_path)
                records.append({
                    "group": source["group"], "algo": source["algo"], "variant": source["variant"],
                    "snapshot_dir": snapshot_dir, "run": run_dir.name, "setting": setting,
                    "seed": seed, "eval_algorithm": algorithm,
                    "csv": csv_path, "result_dir": csv_path.parent,
                })

    available = sorted({record["setting"] for record in records})
    if EVAL_CFGS is not None:
        records = [record for record in records if record["setting"] in set(EVAL_CFGS)]
    if not records:
        raise FileNotFoundError(
            f"Source '{source['algo']} — {source['variant']}': no eval CSVs on setting(s) "
            f"{'all discovered' if EVAL_CFGS is None else EVAL_CFGS} under {snapshot_dir} "
            f"(available: {available or 'none'}) — adjust EVAL_CFGS or the source."
        )
    return records


# --- Group expansion ----------------------------------------------------------------
# One flat source list per active group: the declared members plus one eval-rbc source
# per strategy, all reading the group's single `rbc-ref` snapshot. The other members'
# rule-based runs are deliberately NOT pooled in — within a group they replay the same
# episodes, so pooling would draw each episode several times and shrink the spread. The
# audit cell below checks that assumption instead of assuming it.
def _expand_group(group_name: str) -> list[dict]:
    if group_name not in SNAPSHOT_GROUPS:
        raise KeyError(f"Unknown group {group_name!r} — declared groups: {sorted(SNAPSHOT_GROUPS)}")
    group = SNAPSHOT_GROUPS[group_name]
    sources = [{**member, "group": group_name} for member in group["members"]]

    rbc_ref = group.get("rbc-ref")
    if rbc_ref is not None:
        for strategy in group.get("rbc-strategies", RBC_STRATEGIES):
            sources.append({
                "group": group_name, "path": rbc_ref, "eval-kind": "eval-rbc",
                "algo": "RBC", "variant": strategy, "rbc-strategy": strategy,
            })

    keys = [(source["algo"], source["variant"]) for source in sources]
    if len(keys) != len(set(keys)):
        raise ValueError(f"Group '{group_name}': every source needs a unique (algo, variant) pair; got {keys}")
    return sources


active_groups = list(SNAPSHOT_GROUPS) if ACTIVE_GROUPS is None else list(ACTIVE_GROUPS)
if not active_groups:
    raise ValueError("ACTIVE_GROUPS is empty — name at least one group, or use None for all of them.")

SOURCES = [source for group_name in active_groups for source in _expand_group(group_name)]

# SEEDS normalised to the set of wanted seeds (None = keep every discovered seed): a
# bare int selects one seed, any iterable of ints selects several.
SEED_FILTER: set[int] | None = (
    None if SEEDS is None else {int(seed) for seed in ([SEEDS] if isinstance(SEEDS, int) else SEEDS)}
)
if SEED_FILTER is not None and not SEED_FILTER:
    raise ValueError("SEEDS is empty — use None to keep every discovered seed, or name the seed(s) to keep.")

# Pool a source's eval runs — narrowed to SEED_FILTER when set — but keep only the first
# CSV per (setting, seed): the same checkpoint evaluated with the same seed on the same
# setting replays the identical eval episodes, so a duplicate would double-count them.
# Seed-less CSVs are kept and flagged while no seed filter is active (with one they
# cannot match, so they are dropped). A source left with nothing — eval run still
# pending, or no run on any wanted seed — is skipped with a warning.
eval_records: list[dict] = []
print(f"Repo root    : {REPO_ROOT}")
print(f"Active groups: {active_groups}")
print(f"Eval settings: {'all discovered' if EVAL_CFGS is None else EVAL_CFGS}")
print(f"Eval seeds   : {sorted(SEED_FILTER) if SEED_FILTER is not None else 'all discovered (SEEDS = None)'}")
for source in SOURCES:
    tag = f"{source['group']} / {source['algo']} — {source['variant']}"
    try:
        discovered = sorted(_discover_source_records(source),
                            key=lambda record: (record["setting"], record["run"], record["csv"].name))
    except FileNotFoundError as error:
        print(f"\nWARNING: source '{tag}' skipped — {error}")
        continue

    if SEED_FILTER is not None:
        for record in discovered:
            if record["seed"] is None:
                print(f"note: source '{tag}': dropped {record['run']} / {record['csv'].name} on "
                      f"{record['setting']} (no seed resolvable — cannot match SEEDS)")
        seeds_available = sorted({record["seed"] for record in discovered if record["seed"] is not None})
        discovered = [record for record in discovered if record["seed"] in SEED_FILTER]
        if not discovered:
            print(f"\nWARNING: source '{tag}' skipped — no eval CSVs with a seed in "
                  f"{sorted(SEED_FILTER)} (seeds available: {seeds_available or 'none'}) — "
                  "adjust SEEDS or the source.")
            continue

    kept, seen = [], set()
    for record in discovered:
        if record["seed"] is not None and (record["setting"], record["seed"]) in seen:
            print(f"note: source '{tag}': dropped {record['run']} / {record['csv'].name} on "
                  f"{record['setting']} (duplicate seed {record['seed']})")
            continue
        if record["seed"] is None:
            print(f"WARNING: source '{tag}': no seed resolvable for {record['csv'].name} on "
                  f"{record['setting']} — kept, duplicate eval runs undetectable")
        else:
            seen.add((record["setting"], record["seed"]))
        kept.append(record)
    eval_records.extend(kept)

    strategy = f", strategy={source['rbc-strategy']}" if "rbc-strategy" in source else ""
    print(f"\nSource '{tag}' ({source.get('eval-kind', 'eval-ray')}{strategy}): {source['path']}")
    for record in kept:
        print(f"  {record['setting']:>8}  seed {str(record['seed']):>5}  <-  {record['run']} / "
              f"{record['csv'].name}  [{record['eval_algorithm']}]")

if not eval_records:
    raise FileNotFoundError("No eval CSVs discovered for any source — nothing to plot.")

# Per group: x-axis / hue orders = first-appearance order in that group's source list,
# restricted to the sources that yielded eval results. Which of them actually appear in
# one figure is narrowed again per eval setting (a source may lack results for a single
# setting).
group_labels = [group_name for group_name in active_groups
                if any(record["group"] == group_name for record in eval_records)]
group_algo_labels: dict[str, list[str]] = {}
group_variant_labels: dict[str, list[str]] = {}
for group_name in group_labels:
    present = {(record["algo"], record["variant"]) for record in eval_records if record["group"] == group_name}
    ordered = [source for source in SOURCES if source["group"] == group_name]
    group_algo_labels[group_name] = list(dict.fromkeys(
        source["algo"] for source in ordered if (source["algo"], source["variant"]) in present))
    group_variant_labels[group_name] = list(dict.fromkeys(
        source["variant"] for source in ordered if (source["algo"], source["variant"]) in present))


# Figure order: the flat "default" setting first (if present), then eval_<i> numerically.
def _setting_sort_key(setting: str) -> tuple[int, int]:
    match = re.fullmatch(r"eval_(\d+)", setting)
    return (0, 0) if match is None else (1, int(match.group(1)))


group_setting_labels = {
    group_name: sorted({record["setting"] for record in eval_records if record["group"] == group_name},
                       key=_setting_sort_key)
    for group_name in group_labels
}

# Warn where a source does not cover every setting tabled for its group: those columns
# stay empty for it (e.g. an RBC snapshot whose rule-based run holds fewer held-out
# configs than the RL snapshots do).
for source in SOURCES:
    if source["group"] not in group_labels:
        continue
    covered = {record["setting"] for record in eval_records
               if record["group"] == source["group"] and record["algo"] == source["algo"]
               and record["variant"] == source["variant"]}
    uncovered = [setting for setting in group_setting_labels[source["group"]] if setting not in covered]
    if covered and uncovered:
        print(f"WARNING: source '{source['group']} / {source['algo']} — {source['variant']}' has no eval "
              f"results on {uncovered} — those columns stay empty for it.")

print("\nTables to export (one set per group; the settings are the column blocks):")
for group_name in group_labels:
    settings = [f"{setting} ({EVAL_CFG_LABELS[setting]})" if setting in EVAL_CFG_LABELS else setting
                for setting in group_setting_labels[group_name]]
    print(f"  {group_name:>14}: algos={group_algo_labels[group_name]} settings={settings}")

Repo root    : /hkfs/home/haicore/iai/dj0397/AdvBuildingGym
Active groups: ['base', 'v2']
Eval settings: all discovered
Eval seeds   : [42]

Source 'base / PPO — base' (eval-ray): snapshots/20260721_093047_sta_lin_battery_EV_ppo
   default  seed    42  <-  eval_20260906_235909 / eval_checkpoint_000058_20260907_000020.csv  [ppo]

Source 'base / SAC — base' (eval-ray): snapshots/20260721_103047_sta_lin_battery_EV_sac
   default  seed    42  <-  eval_20260906_235918 / eval_checkpoint_000348_20260907_000350.csv  [sac]

Source 'v2 / PPO — v2' (eval-ray): snapshots/20260721_101901_sta_lin_battery_EV_ppo_v2
   default  seed    42  <-  eval_20260906_235913 / eval_checkpoint_000058_20260907_000150.csv  [ppo]

Source 'v2 / SAC — v2' (eval-ray): snapshots/20260721_111901_sta_lin_battery_EV_sac_v2
   default  seed    42  <-  eval_20260906_235920 / eval_checkpoint_000348_20260907_000551.csv  [sac]

Tables to export (one set per group; the settings are the column blocks):
            base: algos=['P

## Group audit — is this group actually comparable?

Nothing enforces that a hand-written group holds comparable snapshots, so this cell checks it
against the snapshots themselves rather than against their directory names:

- **reward stack** — the `rewards:` block of each snapshot's own trial YAML
  (`code/<manifest.source_trial_path>`). Members that disagree may still share a figure, but their
  `total_reward` columns are on different scales and the cell says so.
- **eval dataset** — the digest of the eval data schedule the trial points at
  (`code/<data_schedule.eval>`). Members that disagree replay a different episode population, so
  *none* of the metrics — nor the RBC baselines — are comparable, and that is reported as an error
  rather than a warning.
- **RBC agreement** — every member's own `runs/eval_rbc_*` run is loaded and compared, per
  strategy, against the group's `rbc-ref`. This is what justifies drawing only the reference:
  if the episodes really are the same population the differences are 0.

The cell also records each snapshot's `control_step` and `EPISODE_LENGTH` from its trial YAML;
the BES analysis further down needs the step length in hours and takes it from here instead of
assuming 5 minutes.

In [4]:
SNAPSHOT_METRICS_FOR_RBC_AUDIT = ("total_reward", "cum_E_kWh", "cum_price_EUR")


def _trial_config(snapshot_dir: Path) -> dict:
    """The snapshot's own trial YAML, located through its manifest's source_trial_path."""
    manifest = json.loads((snapshot_dir / "manifest.json").read_text())
    trial_path = snapshot_dir / "code" / manifest["source_trial_path"]
    return yaml.safe_load(trial_path.read_text())


def _digest(payload: str) -> str:
    return hashlib.sha256(payload.encode()).hexdigest()[:10]


def _snapshot_fingerprint(snapshot_dir: Path) -> dict:
    """Reward-stack digest, eval-dataset digest and env timing of one snapshot."""
    trial = _trial_config(snapshot_dir)
    eval_schedule_path = snapshot_dir / "code" / trial["data_schedule"]["eval"]
    env_meta = trial.get("env_meta", {})
    return {
        "algorithm": trial.get("algorithm"),
        "reward_digest": _digest(json.dumps(trial.get("rewards"), sort_keys=True)),
        "reward_classes": [entry["class_name"] for entry in trial.get("rewards", [])],
        "eval_data_digest": _digest(eval_schedule_path.read_text()),
        "control_step_s": int(env_meta.get("control_step", 300)),
        "episode_length": int(env_meta.get("EPISODE_LENGTH", 288)),
    }


# One fingerprint per snapshot that contributed records (RBC sources share the ref's).
snapshot_fingerprints: dict[Path, dict] = {}
for record in eval_records:
    if record["snapshot_dir"] not in snapshot_fingerprints:
        snapshot_fingerprints[record["snapshot_dir"]] = _snapshot_fingerprint(record["snapshot_dir"])

fingerprint_rows = []
for group_name in group_labels:
    for source in SOURCES:
        if source["group"] != group_name or "rbc-strategy" in source:
            continue
        snapshot_dir = _snapshot_dir(source["path"])
        fingerprint = snapshot_fingerprints.get(snapshot_dir)
        if fingerprint is None:
            continue
        fingerprint_rows.append({
            "group": group_name, "algo": source["algo"], "variant": source["variant"],
            "snapshot": snapshot_dir.name, **fingerprint,
        })

fingerprints = pd.DataFrame(fingerprint_rows)
for group_name in group_labels:
    members = fingerprints[fingerprints["group"] == group_name]
    if members.empty:
        continue
    print(f"\n=== group '{group_name}' — {SNAPSHOT_GROUPS[group_name].get('label', '')}")
    for _, row in members.iterrows():
        print(f"  {row['algo']:>4} {row['variant']:<20} rewards={row['reward_digest']} "
              f"eval_data={row['eval_data_digest']}  {row['snapshot']}")
    if members["eval_data_digest"].nunique() > 1:
        print("  ERROR: members evaluate on DIFFERENT eval datasets — the episodes are a different "
              "population, so no metric and no RBC baseline is comparable inside this group. Split it.")
    if members["reward_digest"].nunique() > 1:
        print("  WARNING: members score under DIFFERENT reward stacks — 'total_reward' is not "
              "comparable across them (cum_E_kWh / cum_price_EUR still are).")
        for digest, stacks in members.groupby("reward_digest")["reward_classes"]:
            print(f"    {digest}: {sorted({tuple(stack) for stack in stacks})}")
    if members["control_step_s"].nunique() > 1 or members["episode_length"].nunique() > 1:
        print("  WARNING: members differ in control_step / EPISODE_LENGTH — per-step analyses below "
              "use each snapshot's own value, but the episodes span different horizons.")


# --- RBC agreement ------------------------------------------------------------------
# Load every member's own rule-based run for the group's strategies and compare it, per
# strategy and metric, against the group's rbc-ref. Only the reference is drawn; this is
# the evidence that the others carry the same episodes.
def _rbc_episodes(snapshot_path: str, strategy: str, group_name: str) -> pd.DataFrame | None:
    source = {"group": group_name, "path": snapshot_path, "eval-kind": "eval-rbc",
              "algo": "RBC", "variant": strategy, "rbc-strategy": strategy}
    try:
        records = sorted(_discover_source_records(source),
                         key=lambda record: (record["setting"], record["run"], record["csv"].name))
    except (FileNotFoundError, ValueError):
        return None
    if SEED_FILTER is not None:
        records = [record for record in records if record["seed"] in SEED_FILTER]
    if not records:
        return None
    frames = []
    for record in records[:1]:  # first (setting, seed) only — same dedup rule as above
        frame = pd.read_csv(record["csv"])
        frame["setting"] = record["setting"]
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


audit_rows = []
for group_name in group_labels:
    group = SNAPSHOT_GROUPS[group_name]
    reference_path = group.get("rbc-ref")
    if reference_path is None:
        continue
    strategies = group.get("rbc-strategies", RBC_STRATEGIES)
    member_paths = [member["path"] for member in group["members"] if member["path"] != reference_path]
    for strategy in strategies:
        reference = _rbc_episodes(reference_path, strategy, group_name)
        if reference is None:
            continue
        for member_path in member_paths:
            other = _rbc_episodes(member_path, strategy, group_name)
            if other is None or len(other) != len(reference):
                audit_rows.append({"group": group_name, "strategy": strategy,
                                   "member": _snapshot_dir(member_path).name,
                                   **{metric: np.nan for metric in SNAPSHOT_METRICS_FOR_RBC_AUDIT}})
                continue
            audit_rows.append({
                "group": group_name, "strategy": strategy, "member": _snapshot_dir(member_path).name,
                **{metric: float(np.abs(reference[metric].to_numpy() - other[metric].to_numpy()).max())
                   for metric in SNAPSHOT_METRICS_FOR_RBC_AUDIT if metric in reference and metric in other},
            })

RBC_AUDIT_TOLERANCE = 1e-6  # episodes replayed from the same config match to the bit

rbc_audit = pd.DataFrame(audit_rows)
if rbc_audit.empty:
    print("\nRBC audit: no non-reference member carries a rule-based run — nothing to cross-check.")
else:
    numeric = [metric for metric in SNAPSHOT_METRICS_FOR_RBC_AUDIT if metric in rbc_audit.columns]
    # An all-NaN row means the member has no comparable rule-based run at all — that is a
    # missing cross-check, NOT a divergence, so it must not be reported as one.
    comparable = rbc_audit[numeric].notna().any(axis=1)
    print("\nRBC audit — max |reference - member| per episode (0 = identical episodes):")
    for group_name, block in rbc_audit.groupby("group"):
        checked = block[comparable.reindex(block.index, fill_value=False)]
        if checked.empty:
            members = sorted(set(block["member"]))
            print(f"  {group_name}: not cross-checked — no rule-based run on {members}; "
                  "the reference is used as-is.")
            continue
        worst = checked[numeric].max()
        print(f"  {group_name}: " + ", ".join(f"{metric} {worst[metric]:.3g}" for metric in numeric)
              + f"  ({len(checked)} of {len(block)} member/strategy pairs cross-checked)")
        divergent = checked[(checked[numeric] > RBC_AUDIT_TOLERANCE).any(axis=1)]
        for _, row in divergent.iterrows():
            print(f"    WARNING: {row['strategy']} differs on {row['member']} — "
                  + ", ".join(f"{metric}={row[metric]:.4g}" for metric in numeric)
                  + " — the group may not be comparable.")
        missing = block[~comparable.reindex(block.index, fill_value=False)]
        for _, row in missing.iterrows():
            print(f"    note: {row['strategy']} not cross-checked — {row['member']} has no "
                  "comparable rule-based run.")
rbc_audit


=== group 'base' — EVChargingSessionReward (charge_progress_scale 0.5) — held-out EV profiles
   PPO base                 rewards=d8d8b5d5f6 eval_data=a42f59e848  20260721_093047_sta_lin_battery_EV_ppo
   SAC base                 rewards=d8d8b5d5f6 eval_data=a42f59e848  20260721_103047_sta_lin_battery_EV_sac

=== group 'v2' — EVChargingSessionReward (charge_progress_scale 10.0) — held-out EV profiles
   PPO v2                   rewards=7fe726feef eval_data=a42f59e848  20260721_101901_sta_lin_battery_EV_ppo_v2
   SAC v2                   rewards=7fe726feef eval_data=a42f59e848  20260721_111901_sta_lin_battery_EV_sac_v2

RBC audit: no non-reference member carries a rule-based run — nothing to cross-check.


""


## Output naming

Everything saved here lands next to the notebook as `<group>_<date>_<time>_out_<i>.<file format>`:
one timestamp per notebook run, `out_index` counting up once per exported table (its CSV and
`.tex` share that index), so no run overwrites an earlier one and the group is part of the
filename rather than something to remember per run.

In [5]:
# VS Code exposes the notebook path; a plain Jupyter kernel starts in the notebook's directory.
OUT_DIR = Path(globals()["__vsc_ipynb_file__"]).parent if "__vsc_ipynb_file__" in globals() else Path.cwd()
OUT_RUN_STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
out_index = 0  # incremented once per exported table; reset by re-running this cell


def next_out_stem(tag: str | None = None) -> Path:
    """Next '[<tag>_]<date>_<time>_out_<i>' path stem in this notebook's directory.

    One index per exported table — its CSV and LaTeX body differ only in the suffix."""
    global out_index
    out_index += 1
    prefix = f"{tag}_" if tag else ""
    return OUT_DIR / f"{prefix}{OUT_RUN_STAMP}_out_{out_index}"

## Load per-episode metrics

The kept CSVs are concatenated into one frame with a row per eval episode and the metrics as
columns, each row tagged with its `group`, `algo`, training `variant`, `eval_cfg` (the setting)
and eval `seed`. The preview below counts the episodes behind each table cell.

In [6]:
frames = []
for record in eval_records:
    csv_frame = pd.read_csv(record["csv"])
    present = [metric for metric in METRICS if metric in csv_frame.columns]
    missing = [metric for metric in METRICS if metric not in csv_frame.columns]
    if missing:
        print(f"note: source '{record['group']} / {record['algo']} — {record['variant']}' "
              f"{record['run']} ({record['setting']}) lacks {missing} — skipped in those tables")
    frame = csv_frame[present].copy()
    frame["group"] = record["group"]
    frame["algo"] = record["algo"]
    frame["variant"] = record["variant"]
    frame["eval_cfg"] = record["setting"]
    frame["seed"] = record["seed"]
    frames.append(frame)

episodes = pd.concat(frames, ignore_index=True)
present_metrics = [metric for metric in METRICS if metric in episodes.columns]


# Reindexes a (group, eval_cfg, algo, variant)-indexed frame into table order.
def _in_figure_order(frame: pd.DataFrame) -> pd.DataFrame:
    ordered = []
    for group_name in group_labels:
        for setting in group_setting_labels[group_name]:
            for algo in group_algo_labels[group_name]:
                for variant in group_variant_labels[group_name]:
                    key = (group_name, setting, algo, variant)
                    if key in frame.index:
                        ordered.append(key)
    return frame.loc[ordered]


# The (algo, variant) row order of one group's tables.
def _row_order(group_name: str, present: set[tuple[str, str]]) -> list[tuple[str, str]]:
    return [(algo, variant)
            for algo in group_algo_labels[group_name]
            for variant in group_variant_labels[group_name]
            if (algo, variant) in present]


# Episodes behind each table cell: rows (group, algo, variant), columns the eval settings.
episodes.groupby(["group", "algo", "variant", "eval_cfg"]).size().unstack("eval_cfg")

eval_cfg            default
group algo variant         
base  PPO  base          10
      SAC  base          10
v2    PPO  v2            10
      SAC  v2            10

In [7]:
_STATS = ("count", "mean", "std", "median", "min", "max")


# LaTeX-escape a label (the RBC strategy names carry underscores).
def _latex_escape(text: str) -> str:
    return text.replace("&", r"\&").replace("%", r"\%").replace("_", r"\_")


# The variant column of a .tex body: TEX_VARIANT_LABELS values are raw LaTeX, unmapped
# variants fall back to their raw label, escaped.
def _variant_cell(variant: str) -> str:
    return TEX_VARIANT_LABELS.get(variant, _latex_escape(str(variant)))


# One numeric cell: "$mean \pm std$"; bare mean when std is undefined (single-episode
# cell), "--" when the (row, column) combination has no episodes.
def _latex_cell(value: float, spread: float | None = None, decimals: int = TEX_DECIMALS) -> str:
    if pd.isna(value):
        return "--"
    if spread is not None and pd.notna(spread):
        return f"${value:.{decimals}f} \\pm {spread:.{decimals}f}$"
    return f"${value:.{decimals}f}$"


# A booktabs body from already-formatted cells: `rows` is a list of
# ((algo, variant), [cell, ...]). \midrule separates the algorithm blocks and the
# algorithm label is printed on its block's first row only. The file ends with
# \bottomrule ON PURPOSE: LaTeX inserts file-hook tokens at the end of an \input{}ed
# file, and after the last row's \\ those tokens start a new cell — a \bottomrule placed
# after \input{} in the skeleton therefore dies with "Misplaced \noalign". Inside the
# file it is safe; \end{tabular} may follow \input{} directly.
def _latex_body(header_comments: list[str], rows: list[tuple[tuple[str, str], list[str]]],
                column_count: int) -> str:
    lines = [
        *(f"% {comment}" for comment in header_comments),
        "% incl. trailing \\bottomrule (a \\bottomrule AFTER \\input{} would be 'Misplaced \\noalign').",
        "% Verified skeleton:",
        f"%   \\begin{{tabular}}{{ll*{{{column_count}}}{{c}}}}",
        "%     \\toprule",
        "%     Algorithm & Variant & ... column heads ... \\\\",
        "%     \\midrule",
        "%     \\input{<this file>}",
        "%   \\end{tabular}",
    ]
    previous_algo = None
    for (algo, variant), cells in rows:
        if previous_algo is not None and algo != previous_algo:
            lines.append(r"\midrule")
        label_cells = [_latex_escape(str(algo)) if algo != previous_algo else "", _variant_cell(variant)]
        lines.append(" & ".join(label_cells + cells) + r" \\")
        previous_algo = algo
    lines.append(r"\bottomrule")
    return "\n".join(lines) + "\n"


# Writes one table's CSV (full precision) and .tex body next to the notebook, sharing an index.
def _export(group_name: str, name: str, frame: pd.DataFrame, latex_body: str) -> None:
    stem = next_out_stem(tag=group_name)
    frame.to_csv(f"{stem}.csv")
    Path(f"{stem}.tex").write_text(latex_body)
    print(f"saved {Path(stem).name}.{{csv,tex}}  ({group_name}, {name})")

## Per-metric statistics tables — display and export

One table per (group, metric): rows = `(algorithm, variant)`, columns = `(eval setting, stat)`.
The cell below displays each table rounded to 3 decimals and writes the CSV (full precision,
`<setting>_<stat>` header) plus the LaTeX body (one `mean ± std` and one `median` column per
setting, `TEX_DECIMALS` decimals) next to this notebook.

The `total_reward` table of a group is only meaningful because the audit above confirmed a single
reward stack inside it — a mixed group would put returns of different scales in one column.

In [8]:
# Wide per-(group, metric) table: rows (algo, variant), columns (eval setting, stat).
def _metric_stats_wide(group_name: str, metric: str) -> pd.DataFrame:
    data = episodes[episodes["group"] == group_name]
    stats = data.groupby(["algo", "variant", "eval_cfg"])[metric].agg(list(_STATS))
    wide = stats.unstack("eval_cfg").swaplevel(axis=1)
    settings = group_setting_labels[group_name]
    wide = wide.reindex(columns=pd.MultiIndex.from_product([settings, _STATS]))
    return wide.reindex(_row_order(group_name, set(zip(data["algo"], data["variant"]))))


for group_name in group_labels:
    print(f"\n===== group '{group_name}' — {SNAPSHOT_GROUPS[group_name].get('label', '')}")
    settings = group_setting_labels[group_name]
    for metric in present_metrics:
        wide = _metric_stats_wide(group_name, metric)
        export_frame = wide.copy()
        export_frame.columns = [f"{setting}_{stat}" for setting, stat in wide.columns]
        rows = [(key, [cell
                       for setting in settings
                       for cell in (_latex_cell(row[(setting, "mean")], row[(setting, "std")]),
                                    _latex_cell(row[(setting, "median")]))])
                for key, row in wide.iterrows()]
        body = _latex_body(
            [f"Auto-generated by sps_eval_stats_table_algo_cmp.ipynb — booktabs body for "
             f"'{metric}', group '{group_name}'.",
             "Columns: algorithm & variant"
             + "".join(f" & {setting} mean+-std & {setting} median" for setting in settings) + ".",
             "Full precision, count, min and max: see the CSV with the same stem."],
            rows, 2 * len(settings))
        _export(group_name, metric, export_frame, body)
        print(f"\n=== {group_name} / {metric} ===")
        display(wide.round(3))


===== group 'base' — EVChargingSessionReward (charge_progress_scale 0.5) — held-out EV profiles
saved base_20260925_133745_out_1.{csv,tex}  (base, total_reward)

=== base / total_reward ===


default                                         
               count    mean     std  median      min     max
algo variant                                                 
PPO  base         10 -38.688  16.574 -33.769  -80.772 -23.629
SAC  base         10 -52.841  45.216 -32.641 -120.721 -12.978

saved base_20260925_133745_out_2.{csv,tex}  (base, cum_E_kWh)

=== base / cum_E_kWh ===


default                                        
               count    mean     std  median     min     max
algo variant                                                
PPO  base         10  12.899  39.879  16.738 -71.690  69.156
SAC  base         10   8.055  17.710  10.294 -14.503  42.782

saved base_20260925_133745_out_3.{csv,tex}  (base, cum_price_EUR)

=== base / cum_price_EUR ===


...

default                                        
               count   mean     std  median      min     max
algo variant                                                
PPO  v2           10 -48.29  62.288 -30.509 -120.381  40.509
SAC  v2           10 -34.20  38.454 -32.109 -113.314  42.450

saved v2_20260925_133745_out_5.{csv,tex}  (v2, cum_E_kWh)

=== v2 / cum_E_kWh ===


default                                        
               count    mean     std  median     min     max
algo variant                                                
PPO  v2           10  16.652  24.285  15.642 -21.696  53.632
SAC  v2           10  -3.182  45.742  -3.358 -73.545  63.963

saved v2_20260925_133745_out_6.{csv,tex}  (v2, cum_price_EUR)

=== v2 / cum_price_EUR ===


default                                   
               count   mean    std median    min    max
algo variant                                           
PPO  v2           10  0.968  1.150  1.006 -0.930  2.570
SAC  v2           10  1.382  3.612  1.634 -3.564  6.761

## Per-step trajectories — the input for the EV and BES analyses

The eval CSVs carry only per-episode totals, so the EV-session counts and the battery summary are
computed from the **per-step trajectories** written next to every eval CSV
(`<result dir>/trajectories/<episode>_trajectory.json`). Only the runs that survived the seed and
duplicate filtering above are read, so the trajectories always belong to the episodes drawn in
the figures — and only for the **active groups**.

The columns used (all verified present in these snapshots' trajectory JSONs):

| Column | Meaning |
|---|---|
| `state.s_evc_connected` | 1 while the charger holds an EV; the charger applies the schedule's connect/disconnect in `exec_action`, so this flips one step after `ctxt_ev_schedule_max_cap_kWh` |
| `state.s_evc_soc`, `state.s_evc_target_soc` | EV SoC and its session target — both **zeroed on detach** (`LinearEVCharger.set_ev_connected` clears `soc`, and `target_soc` returns 0.0 with no `ev_spec`), so a session's verdict must be read from its **last connected** row |
| `state.s_evc_session_target_feasible` | snapshotted at connect: can the target still be reached at full charging rate? |
| `state.s_battery_soc` | battery SoC |
| `state.s_sim_time` | episode progress; hour of day = `s_sim_time × 24` for these one-day episodes |
| `power_breakdown.battery` | the step's battery power as `production − consumption` (`extract_trajectory_from_infos`), i.e. **positive = discharge**, negative = charge |

Row 0 of every column is the reset row (zero action, zero power), so step `t ≥ 1` acted under the
state of row `t − 1` — which is what the hour-of-day binning below uses.

In [9]:
# Reading one group's trajectories is ~80 JSON files of ~0.4 MB; cached so re-running
# the analysis cells below is instant.
@lru_cache(maxsize=None)
def _load_trajectory(path: str) -> dict:
    return json.loads(Path(path).read_text())["trajectory"]


def _column(trajectory: dict, section: str, key: str) -> np.ndarray | None:
    """One columnar trajectory series as a float array, or None when absent."""
    values = trajectory.get(section, {}).get(key)
    return None if values is None else np.asarray(values, dtype=float).ravel()


# Every trajectory JSON of the kept eval records, in figure order. `result_dir` is the
# directory the eval CSV sits in, and the eval writers put `trajectories/` right beside
# it (both for the flat STA layout and the per-config GEN dirs).
def _trajectory_records() -> list[dict]:
    records = []
    for record in eval_records:
        trajectory_dir = record["result_dir"] / "trajectories"
        paths = sorted(trajectory_dir.glob("*_trajectory.json"),
                       key=lambda path: int(path.name.split("_")[0]))
        if not paths:
            print(f"WARNING: no trajectories under {trajectory_dir} — "
                  f"'{record['group']} / {record['algo']} — {record['variant']}' is absent "
                  "from the EV and BES analyses.")
            continue
        for path in paths:
            records.append({
                "group": record["group"], "algo": record["algo"], "variant": record["variant"],
                "eval_cfg": record["setting"], "seed": record["seed"],
                "snapshot_dir": record["snapshot_dir"],
                "episode": int(path.name.split("_")[0]), "path": path,
            })
    return records


trajectory_records = _trajectory_records()
print(f"{len(trajectory_records)} eval-episode trajectories over "
      f"{len({record['path'].parent for record in trajectory_records})} result directories")

40 eval-episode trajectories over 4 result directories


## EV charging success

One charging **session** is a maximal run of connected steps. It ends when the charger releases
the EV — the `s_evc_connected` 1 → 0 edge — and that is the step `EVChargingSessionReward` issues
its verdict on:

```python
if was_connected and not now_connected:
    achieved_soc = float(state["s_evc_soc"][0])          # pre-detach row
    target_soc = session_target_soc(info)                # latched, not cleared on detach
    return self.weight * (self.success_reward
                          if achieved_soc >= target_soc - self.disconnect_soc_tolerance
                          else self.failure_penalty)
```

so the table below reads `s_evc_soc` and `s_evc_target_soc` from the session's **last connected**
row and reports both columns asked for:

- **met@tol** — `achieved ≥ target − EV_SOC_TOLERANCE`, reproducing the verdict the policy was
  rewarded on (`disconnect_soc_tolerance = 0.05` in every trial YAML of these snapshots);
- **met@strict** — `achieved ≥ target`, with no tolerance;
- **ignored** — while plugged in, the SoC never moves `EV_IGNORED_SOC_CHANGE` (5 % SoC) or more
  away from the SoC the EV arrived with: the policy left the car essentially untouched.

The arrival SoC is not the first connected row. On connect, `LinearEVCharger.exec_action` sets
`soc = start_soc` and then charges in the same step, so that row already holds one step of
charging. It is read instead from `ctxt_ev_schedule_start_soc` on the row *before* the connect:
the value the charger copies. V2G is enabled in these trials (`v2g_enabled: true`), so the SoC
can also fall. The largest distance from the arrival SoC in either direction is therefore used,
not the net change. An ignored session can still count as met, e.g. when the EV arrives at or
above its target.

A session still connected at the horizon gets **no** verdict (the reward never fires either); it
is counted in the `open` column and excluded from the rates and from `ignored`. `infeasible` counts sessions whose
`s_evc_session_target_feasible` was 0 — the target was already unreachable at full charging rate
when the EV plugged in, so the miss is not the policy's doing.

Note on the rule-based rows: `RuleBasedStrategy` only ever emits battery actions
(`_battery_action` / `zero_action` in `adv_building_gym/rbc_strats/base.py`), so `a_lin_ev_charger`
stays 0 for every RBC strategy and no session can be met. Their 0 % is the *no-EV-control*
reference, not a failed controller.

In [10]:
# One row per charging session of one episode. The charger zeroes s_evc_soc and
# s_evc_target_soc on detach, so achieved SoC and target are read from the session's
# LAST CONNECTED row — the same row EVChargingSessionReward judges as `state` (s).
def ev_sessions(trajectory: dict) -> list[dict]:
    connected = _column(trajectory, "state", "s_evc_connected")
    soc = _column(trajectory, "state", "s_evc_soc")
    target = _column(trajectory, "state", "s_evc_target_soc")
    feasible = _column(trajectory, "state", "s_evc_session_target_feasible")
    start_soc = _column(trajectory, "state", "ctxt_ev_schedule_start_soc")
    if connected is None or soc is None or target is None:
        return []

    plugged = connected > 0.5
    indices = np.flatnonzero(plugged)
    if indices.size == 0:
        return []
    segments = np.split(indices, np.flatnonzero(np.diff(indices) != 1) + 1)

    sessions = []
    for segment in segments:
        first, last = int(segment[0]), int(segment[-1])
        achieved, wanted = float(soc[last]), float(target[last])
        # On connect the charger sets soc = start_soc and charges in that same step, so the
        # arrival SoC is the schedule's start SoC on the row BEFORE the connect: the value
        # LinearEVCharger._check_schedule copies. Without that column the first connected
        # row is the closest stand-in, missing that first step's charge.
        arrival = float(start_soc[first - 1]) if start_soc is not None and first > 0 else float(soc[first])
        # Largest move away from the arrival SoC in either direction (V2G can lower it).
        soc_max_change = float(np.abs(soc[first:last + 1] - arrival).max())
        sessions.append({
            "start_step": first,
            "end_step": last,
            "steps_connected": len(segment),
            "achieved_soc": achieved,
            "target_soc": wanted,
            "soc_gap": max(0.0, wanted - achieved),
            "arrival_soc": arrival,
            "soc_max_change": soc_max_change,
            "ignored": bool(soc_max_change < EV_IGNORED_SOC_CHANGE),
            "met_tolerance": bool(achieved >= wanted - EV_SOC_TOLERANCE),
            "met_strict": bool(achieved >= wanted),
            "feasible": bool(feasible[last] > 0.5) if feasible is not None else True,
            # No disconnect edge inside the episode -> no verdict is ever issued.
            "open_at_horizon": last == plugged.size - 1,
        })
    return sessions


ev_session_rows = []
for record in trajectory_records:
    trajectory = _load_trajectory(str(record["path"]))
    for session in ev_sessions(trajectory):
        ev_session_rows.append({
            "group": record["group"], "algo": record["algo"], "variant": record["variant"],
            "eval_cfg": record["eval_cfg"], "seed": record["seed"], "episode": record["episode"],
            **session,
        })

ev_sessions_frame = pd.DataFrame(ev_session_rows)
if ev_sessions_frame.empty:
    raise ValueError("No EV charging sessions found in any trajectory — is this an EV trial?")

_keys = ["group", "eval_cfg", "algo", "variant"]
# Eval episodes per strip, counted from the trajectories rather than from the sessions:
# an episode whose eval day schedules no EV at all (the `_eval_ch` groups have some)
# contributes no session but is still an evaluated episode.
_strip_episodes = pd.DataFrame(trajectory_records).groupby(_keys)["episode"].nunique()
# Sessions with a verdict; the ones still connected at the horizon are counted apart.
# `infeasible` is materialised as a column so it sums like the other counts instead of
# needing a per-group Python lambda.
_closed = ev_sessions_frame[~ev_sessions_frame["open_at_horizon"]].assign(
    infeasible=lambda frame: ~frame["feasible"])
_closed_by_strip = _closed.groupby(_keys)

ev_success = pd.DataFrame({
    "episodes": _strip_episodes,
    "episodes_with_session": ev_sessions_frame.groupby(_keys)["episode"].nunique(),
    "sessions": _closed_by_strip.size(),
    "met@tol": _closed_by_strip["met_tolerance"].sum(),
    "rate@tol": _closed_by_strip["met_tolerance"].mean(),
    "met@strict": _closed_by_strip["met_strict"].sum(),
    "rate@strict": _closed_by_strip["met_strict"].mean(),
    "mean_soc_gap": _closed_by_strip["soc_gap"].mean(),
    "max_soc_gap": _closed_by_strip["soc_gap"].max(),
    "infeasible": _closed_by_strip["infeasible"].sum(),
    "ignored": _closed_by_strip["ignored"].sum(),
    "open": ev_sessions_frame.groupby(_keys)["open_at_horizon"].sum(),
}).reindex(_strip_episodes.index)
# A strip whose episodes hold no session at all: zero counts, undefined rates.
ev_success[["episodes_with_session", "sessions", "met@tol", "met@strict", "infeasible", "ignored", "open"]] = (
    ev_success[["episodes_with_session", "sessions", "met@tol", "met@strict", "infeasible", "ignored", "open"]]
    .fillna(0).astype(int)
)
ev_success = _in_figure_order(ev_success)
ev_success.round(3)

episodes  episodes_with_session  sessions  \
group eval_cfg algo variant                                              
base  default  PPO  base           10                     10        18   
               SAC  base           10                     10        18   
v2    default  PPO  v2             10                     10        18   
               SAC  v2             10                     10        18   

                             met@tol  rate@tol  met@strict  rate@strict  \
group eval_cfg algo variant                                               
base  default  PPO  base           8     0.444           8        0.444   
               SAC  base           7     0.389           6        0.333   
v2    default  PPO  v2             6     0.333           6        0.333   
               SAC  v2             8     0.444           6        0.333   

                             mean_soc_gap  max_soc_gap  infeasible  ignored  \
group eval_cfg algo variant                                                   
base  default  PPO  base            0.233        0.570           0        6   
               SAC  base            0.258        0.535           0        5   
v2    default  PPO  v2              0.202        0.550           0        2   
               SAC  v2              0.128        0.438           0        1   

                             open  
group eval_cfg algo variant        
base  default  PPO  base        0  
               SAC  base        0  
v2    default  PPO  v2          0  
               SAC  v2          0

### Export the EV success table

One table per group: rows `(algorithm, variant)`, the columns of `ev_success` above. Only the
`default` / `eval_<i>` settings the group actually holds appear; with a single setting (the STA
layout of these snapshots) a group yields one block of rows. CSV at full precision, `.tex` with
the counts as integers and the rates as percentages.

In [11]:
EV_TEX_COLUMNS = [
    ("sessions", "sessions", 0),
    ("met@tol", "met @tol", 0),
    ("rate@tol", "rate @tol [\\%]", 1),
    ("met@strict", "met strict", 0),
    ("rate@strict", "rate strict [\\%]", 1),
    ("mean_soc_gap", "mean SoC gap", 3),
    ("infeasible", "infeasible", 0),
    ("ignored", "ignored", 0),
]

for group_name in group_labels:
    block = ev_success.loc[group_name]
    for setting in group_setting_labels[group_name]:
        if setting not in block.index.get_level_values("eval_cfg"):
            continue
        table = block.loc[setting]
        rows = []
        for key, row in table.iterrows():
            cells = []
            for column, _, decimals in EV_TEX_COLUMNS:
                value = row[column] * 100 if column.startswith("rate") else row[column]
                cells.append(_latex_cell(value, decimals=decimals))
            rows.append((key, cells))
        body = _latex_body(
            [f"Auto-generated by sps_eval_stats_table_algo_cmp.ipynb — EV charging success, "
             f"group '{group_name}', setting '{setting}'.",
             "A session is met when the SoC at the disconnect step is >= target - "
             f"{EV_SOC_TOLERANCE} (met @tol, the verdict EVChargingSessionReward pays out on)",
             "or >= target (met strict). Sessions still connected at the horizon get no verdict "
             "and are counted in the CSV's 'open' column.",
             f"'ignored' = sessions whose SoC never moves >= {EV_IGNORED_SOC_CHANGE} away from the "
             "arrival SoC while plugged in (charging or V2G).",
             "Columns: algorithm & variant"
             + "".join(f" & {head}" for _, head, _ in EV_TEX_COLUMNS) + "."],
            rows, len(EV_TEX_COLUMNS))
        _export(group_name, f"EV success ({setting})", table, body)
        print(f"\n=== {group_name} / EV charging success ({setting}) ===")
        display(table.round(3))

saved base_20260925_133745_out_7.{csv,tex}  (base, EV success (default))

=== base / EV charging success (default) ===


,,episodes,episodes_with_session,sessions,met@tol,rate@tol,met@strict,rate@strict,mean_soc_gap,max_soc_gap,infeasible,ignored,open
algo,variant,,,,,,,,,,,,
PPO,base,10,10,18,8,0.444,8,0.444,0.233,0.570,0,6,0
SAC,base,10,10,18,7,0.389,6,0.333,0.258,0.535,0,5,0


saved v2_20260925_133745_out_8.{csv,tex}  (v2, EV success (default))

=== v2 / EV charging success (default) ===


,,episodes,episodes_with_session,sessions,met@tol,rate@tol,met@strict,rate@strict,mean_soc_gap,max_soc_gap,infeasible,ignored,open
algo,variant,,,,,,,,,,,,
PPO,v2,10,10,18,6,0.333,6,0.333,0.202,0.550,0,2,0
SAC,v2,10,10,18,8,0.444,6,0.333,0.128,0.438,0,1,0


## BES management summary

How the battery is actually used, per strip. The step's battery power comes from
`power_breakdown.battery`, which `extract_trajectory_from_infos` writes as
`production − consumption` — so **positive = discharge**, negative = charge — and the step energy
is that power times the snapshot's own `control_step` in hours.

Per episode:

| Column | Meaning |
|---|---|
| `charged_kWh` / `discharged_kWh` | energy in / out over the episode |
| `charge_ratio` | `charged_kWh / discharged_kWh` — 0 means the battery is only emptied |
| `early_share` | share of the episode's total `|battery power|` spent before `EARLY_HOURS_END` |
| `dir_changes` | charge ↔ discharge sign flips, ignoring power below `POWER_DEADBAND_KW` |
| `start_soc` / `end_soc` / `min_soc` / `max_soc` | SoC trajectory endpoints and extremes |
| `soc_recovery` | how far the SoC climbs back **after** its lowest point |

### The two flags

Both readings of *"the BES does no more than discharge at the beginning of the day"* are reported
side by side, so a disagreement is visible instead of hidden behind one threshold:

- **`flag_throughput`** — `early_share ≥ EARLY_SHARE_FLAG` **and** `charge_ratio <
  RECHARGE_RATIO_FLAG`: the activity is concentrated in the early hours *and* the battery is
  barely refilled.
- **`flag_soc_shape`** — `soc_recovery ≤ SOC_RECOVERY_TOLERANCE`: the SoC falls to its minimum and
  never climbs back, whatever the timing.

An episode whose total throughput stays under `IDLE_THROUGHPUT_KWH` is classified **idle** and
flagged by neither — an untouched battery (`do_nothing`) is not a drawdown, and without that guard
its zero recovery would trip `flag_soc_shape`. The `flag_*` columns of the summary are the
**fraction of that strip's episodes** carrying the flag; `idle` is the fraction classified idle.

In [12]:
# One row per episode: how the battery moved. Row 0 of every column is the reset row
# (zero action), so step t >= 1 acted under the state of row t - 1 — which is the row
# the hour-of-day binning reads.
def bes_metrics(trajectory: dict, control_step_s: int) -> dict | None:
    power = _column(trajectory, "power_breakdown", "battery")   # >0 discharge, <0 charge
    soc = _column(trajectory, "state", "s_battery_soc")
    sim_time = _column(trajectory, "state", "s_sim_time")
    if power is None or soc is None or sim_time is None:
        return None

    step_hours = control_step_s / 3600.0
    acting_power = power[1:]
    acting_hour = sim_time[:-1] * 24.0
    discharged = float(np.clip(acting_power, 0.0, None).sum() * step_hours)
    charged = float(np.clip(-acting_power, 0.0, None).sum() * step_hours)

    throughput = np.abs(acting_power)
    total_throughput = float(throughput.sum() * step_hours)
    early_share = (float(throughput[acting_hour < EARLY_HOURS_END].sum() / throughput.sum())
                   if throughput.sum() > 0 else np.nan)
    charge_ratio = charged / discharged if discharged > 0 else np.nan

    directions = np.sign(np.where(np.abs(acting_power) > POWER_DEADBAND_KW, acting_power, 0.0))
    directions = directions[directions != 0.0]
    direction_changes = int((np.diff(directions) != 0).sum()) if directions.size > 1 else 0

    # SoC shape: does the SoC ever climb back after its lowest point of the episode?
    trough = int(np.argmin(soc))
    soc_recovery = float(soc[trough:].max() - soc[trough])

    idle = total_throughput < IDLE_THROUGHPUT_KWH
    return {
        "charged_kWh": charged,
        "discharged_kWh": discharged,
        "throughput_kWh": total_throughput,
        "charge_ratio": charge_ratio,
        "early_share": early_share,
        "dir_changes": direction_changes,
        "start_soc": float(soc[0]),
        "end_soc": float(soc[-1]),
        "min_soc": float(soc.min()),
        "max_soc": float(soc.max()),
        "soc_recovery": soc_recovery,
        "idle": bool(idle),
        # Concentrated early AND barely refilled.
        "flag_throughput": bool(not idle and early_share >= EARLY_SHARE_FLAG
                                and (np.isnan(charge_ratio) or charge_ratio < RECHARGE_RATIO_FLAG)),
        # Drawn down, then flat — the SoC never recovers.
        "flag_soc_shape": bool(not idle and soc_recovery <= SOC_RECOVERY_TOLERANCE),
    }


bes_rows = []
for record in trajectory_records:
    control_step_s = snapshot_fingerprints[record["snapshot_dir"]]["control_step_s"]
    metrics = bes_metrics(_load_trajectory(str(record["path"])), control_step_s)
    if metrics is None:
        continue
    bes_rows.append({
        "group": record["group"], "algo": record["algo"], "variant": record["variant"],
        "eval_cfg": record["eval_cfg"], "seed": record["seed"], "episode": record["episode"],
        **metrics,
    })

bes_episodes = pd.DataFrame(bes_rows)
if bes_episodes.empty:
    raise ValueError("No battery trajectories found — is there a battery in this trial?")

_bes_means = ["charged_kWh", "discharged_kWh", "charge_ratio", "early_share", "dir_changes",
              "start_soc", "end_soc", "min_soc", "max_soc", "soc_recovery"]
# Where the two criteria disagree — the reason both are reported. Derived as a plain
# boolean column so it aggregates with everything else: a groupby-apply would need
# `include_groups`, which only exists in pandas >= 2.2 and is deprecated in that very
# release (pandas/core/groupby/groupby.py: "versionadded:: 2.2.0" / "deprecated:: 2.2.0").
bes_episodes["flags_disagree"] = bes_episodes["flag_throughput"] != bes_episodes["flag_soc_shape"]

# The flag_* / idle / flags_disagree columns are booleans, so their mean is the FRACTION
# of that strip's episodes carrying the flag.
bes_summary = _in_figure_order(bes_episodes.groupby(_keys).agg(
    episodes=("episode", "size"),
    **{column: (column, "mean") for column in _bes_means},
    idle=("idle", "mean"),
    flag_throughput=("flag_throughput", "mean"),
    flag_soc_shape=("flag_soc_shape", "mean"),
    flags_disagree=("flags_disagree", "mean"),
))
bes_summary.round(3)

episodes  charged_kWh  discharged_kWh  \
group eval_cfg algo variant                                          
base  default  PPO  base           10        9.730          12.748   
               SAC  base           10       27.261          16.679   
v2    default  PPO  v2             10        1.197           4.215   
               SAC  v2             10       19.854          18.902   

                             charge_ratio  early_share  dir_changes  \
group eval_cfg algo variant                                           
base  default  PPO  base            0.731        0.189          3.4   
               SAC  base            1.687        0.112          7.4   
v2    default  PPO  v2              0.208        0.687          2.2   
               SAC  v2              1.072        0.105          6.1   

                             start_soc  end_soc  min_soc  max_soc  \
group eval_cfg algo variant                                         
base  default  PPO  base         0.289    0.100      0.1    0.706   
               SAC  base         0.289    0.950      0.1    0.950   
v2    default  PPO  v2           0.289    0.100      0.1    0.303   
               SAC  v2           0.289    0.348      0.1    0.950   

                             soc_recovery  idle  flag_throughput  \
group eval_cfg algo variant                                        
base  default  PPO  base            0.593   0.0              0.0   
               SAC  base            0.850   0.0              0.0   
v2    default  PPO  v2              0.075   0.0              0.5   
               SAC  v2              0.850   0.0              0.0   

                             flag_soc_shape  flags_disagree  
group eval_cfg algo variant                                  
base  default  PPO  base                0.0             0.0  
               SAC  base                0.0             0.0  
v2    default  PPO  v2                  0.3             0.2  
               SAC  v2                  0.0             0.0

In [13]:
# Plain-language verdict per strip, so the summary reads without cross-checking thresholds.
print(f"Flags: throughput = >= {EARLY_SHARE_FLAG:.0%} of |battery power| before "
      f"{EARLY_HOURS_END:.0f}:00 AND charged/discharged < {RECHARGE_RATIO_FLAG:.0%}; "
      f"SoC shape = SoC recovers <= {SOC_RECOVERY_TOLERANCE:.2f} after its minimum; "
      f"idle = throughput < {IDLE_THROUGHPUT_KWH:.2f} kWh.\n")
for (group_name, eval_cfg, algo, variant), row in bes_summary.iterrows():
    if row["idle"] >= 0.5:
        verdict = "battery essentially untouched"
    elif row["flag_throughput"] >= 0.5 and row["flag_soc_shape"] >= 0.5:
        verdict = "DISCHARGE-ONLY EARLY — both criteria agree"
    elif row["flag_throughput"] >= 0.5:
        verdict = "early, barely-refilled throughput (SoC shape disagrees — some recovery happens)"
    elif row["flag_soc_shape"] >= 0.5:
        verdict = "SoC never recovers (but the activity is not concentrated early)"
    else:
        verdict = "active management — charges and discharges through the day"
    print(f"{group_name:<13} {algo:<4} {variant:<26} {verdict}")
    print(f"      charged {row['charged_kWh']:6.2f} kWh, discharged {row['discharged_kWh']:6.2f} kWh "
          f"(ratio {row['charge_ratio']:.2f}), early share {row['early_share']:.2f}, "
          f"{row['dir_changes']:.1f} direction changes, SoC {row['start_soc']:.2f} -> "
          f"{row['end_soc']:.2f} (min {row['min_soc']:.2f}, recovery {row['soc_recovery']:.2f})")
    print(f"      episodes flagged: throughput {row['flag_throughput']:.0%}, "
          f"SoC shape {row['flag_soc_shape']:.0%}, disagreeing {row['flags_disagree']:.0%}, "
          f"idle {row['idle']:.0%}")

Flags: throughput = >= 70% of |battery power| before 8:00 AND charged/discharged < 20%; SoC shape = SoC recovers <= 0.02 after its minimum; idle = throughput < 0.10 kWh.

base          PPO  base                       active management — charges and discharges through the day
      charged   9.73 kWh, discharged  12.75 kWh (ratio 0.73), early share 0.19, 3.4 direction changes, SoC 0.29 -> 0.10 (min 0.10, recovery 0.59)
      episodes flagged: throughput 0%, SoC shape 0%, disagreeing 0%, idle 0%
base          SAC  base                       active management — charges and discharges through the day
      charged  27.26 kWh, discharged  16.68 kWh (ratio 1.69), early share 0.11, 7.4 direction changes, SoC 0.29 -> 0.95 (min 0.10, recovery 0.85)
      episodes flagged: throughput 0%, SoC shape 0%, disagreeing 0%, idle 0%
v2            PPO  v2                         early, barely-refilled throughput (SoC shape disagrees — some recovery happens)
      charged   1.20 kWh, discharged   4.21 kWh

### Export the BES management table

One table per group with the columns of `bes_summary` above. The `flag_*` columns are the
**fraction of that row's eval episodes** carrying the flag, so `1.00` means every episode of the
strip discharges early and never recovers, `0.00` none of them. `idle` is the fraction of episodes
whose battery was essentially untouched — those are flagged by neither criterion.

In [14]:
BES_TEX_COLUMNS = [
    ("charged_kWh", "charged [kWh]", 2),
    ("discharged_kWh", "discharged [kWh]", 2),
    ("charge_ratio", "chg/dis", 2),
    ("early_share", "early share", 2),
    ("dir_changes", "dir.\\ changes", 1),
    ("start_soc", "start SoC", 2),
    ("end_soc", "end SoC", 2),
    ("soc_recovery", "SoC recovery", 2),
    ("flag_throughput", "flag thr.\\ [\\%]", 0),
    ("flag_soc_shape", "flag SoC [\\%]", 0),
]

for group_name in group_labels:
    block = bes_summary.loc[group_name]
    for setting in group_setting_labels[group_name]:
        if setting not in block.index.get_level_values("eval_cfg"):
            continue
        table = block.loc[setting]
        rows = []
        for key, row in table.iterrows():
            cells = []
            for column, _, decimals in BES_TEX_COLUMNS:
                value = row[column] * 100 if column.startswith("flag") else row[column]
                cells.append(_latex_cell(value, decimals=decimals))
            rows.append((key, cells))
        body = _latex_body(
            [f"Auto-generated by sps_eval_stats_table_algo_cmp.ipynb — BES management, "
             f"group '{group_name}', setting '{setting}'.",
             "Battery power is production - consumption, so positive = discharge. Per-episode "
             "values averaged over the eval episodes;",
             f"'flag thr.' = >= {EARLY_SHARE_FLAG:.0%} of |battery power| before "
             f"{EARLY_HOURS_END:.0f}:00 AND charged/discharged < {RECHARGE_RATIO_FLAG:.0%}; "
             f"'flag SoC' = SoC recovers <= {SOC_RECOVERY_TOLERANCE} after its minimum;",
             "both are the PERCENTAGE OF EPISODES carrying the flag. Episodes with a throughput "
             f"below {IDLE_THROUGHPUT_KWH} kWh count as idle (CSV column) and carry neither flag.",
             "Columns: algorithm & variant"
             + "".join(f" & {head}" for _, head, _ in BES_TEX_COLUMNS) + "."],
            rows, len(BES_TEX_COLUMNS))
        _export(group_name, f"BES management ({setting})", table, body)
        print(f"\n=== {group_name} / BES management ({setting}) ===")
        display(table.round(3))

saved base_20260925_133745_out_9.{csv,tex}  (base, BES management (default))

=== base / BES management (default) ===


,,episodes,charged_kWh,discharged_kWh,charge_ratio,early_share,dir_changes,start_soc,end_soc,min_soc,max_soc,soc_recovery,idle,flag_throughput,flag_soc_shape,flags_disagree
algo,variant,,,,,,,,,,,,,,,
PPO,base,10,9.730,12.748,0.731,0.189,3.4,0.289,0.10,0.1,0.706,0.593,0.0,0.0,0.0,0.0
SAC,base,10,27.261,16.679,1.687,0.112,7.4,0.289,0.95,0.1,0.950,0.850,0.0,0.0,0.0,0.0


saved v2_20260925_133745_out_10.{csv,tex}  (v2, BES management (default))

=== v2 / BES management (default) ===


,,episodes,charged_kWh,discharged_kWh,charge_ratio,early_share,dir_changes,start_soc,end_soc,min_soc,max_soc,soc_recovery,idle,flag_throughput,flag_soc_shape,flags_disagree
algo,variant,,,,,,,,,,,,,,,
PPO,v2,10,1.197,4.215,0.208,0.687,2.2,0.289,0.100,0.1,0.303,0.075,0.0,0.5,0.3,0.2
SAC,v2,10,19.854,18.902,1.072,0.105,6.1,0.289,0.348,0.1,0.950,0.850,0.0,0.0,0.0,0.0


## Joint summary table

One table per (group, setting) that puts the headline numbers of the analyses above side by side:
rows `(algorithm, variant)`, columns

| Block | Columns | Source |
|---|---|---|
| return | mean, std, median of `total_reward` | eval CSVs (`episodes`); the eval runner sums the step rewards into it, so it is the episode return |
| cost | mean, std, median of `cum_price_EUR` | eval CSVs |
| energy | mean, std, median of `cum_E_kWh` | eval CSVs |
| EV | `sessions`, `met@tol`, `ignored` | `ev_success`: counts summed over the strip's episodes |
| BES | `charged_kWh`, `discharged_kWh`, `charge_ratio`, `dir_changes`, `min_soc`, `max_soc` | `bes_summary`: per-episode values averaged over the strip's episodes |

- `sessions` counts only the sessions that received a verdict. The ones still connected at the
  horizon are left out, as in `ev_success`, where they appear under `open`. So
  `met@tol / sessions` is the success rate.
- A **successful** session is `met@tol`: the verdict `EVChargingSessionReward` pays out on. Swap
  in `met@strict` in `JOINT_EV_COLUMNS` for the count without the tolerance.
- `ignored` counts the sessions (out of `sessions`) whose SoC never moved `EV_IGNORED_SOC_CHANGE`
  (5 % SoC) or more away from the arrival SoC. The EV section explains how the arrival SoC is read.
- `min_soc` / `max_soc` are each episode's lowest / highest battery SoC, **averaged over the
  episodes**. They are not the extremes over all episodes.

Export: a CSV at full precision, with the header `<metric>_<stat>` followed by the raw column
names. The `.tex` body uses one `mean ± std` and one `median` column per metric (the same as the
per-metric tables), followed by the single-value columns.

In [15]:
# Metric blocks in column order: (episodes column, head used in the .tex comment).
JOINT_METRICS = [
    ("total_reward", "return"),
    ("cum_price_EUR", "cost [EUR]"),
    ("cum_E_kWh", "energy [kWh]"),
]
_JOINT_STATS = ("mean", "std", "median")
# Single-value columns after the metric blocks: (column, LaTeX head, decimals). "met@tol"
# is the verdict EVChargingSessionReward pays out on; "met@strict" drops the tolerance.
JOINT_EV_COLUMNS = [
    ("sessions", "sessions", 0),
    ("met@tol", "met @tol", 0),
    ("ignored", "ignored", 0),
]
JOINT_BES_COLUMNS = [
    ("charged_kWh", "charged [kWh]", 2),
    ("discharged_kWh", "discharged [kWh]", 2),
    ("charge_ratio", "chg/dis", 2),
    ("dir_changes", "dir.\\ changes", 1),
    ("min_soc", "min SoC", 2),
    ("max_soc", "max SoC", 2),
]
_joint_single_columns = JOINT_EV_COLUMNS + JOINT_BES_COLUMNS

joint_metrics = [(metric, head) for metric, head in JOINT_METRICS if metric in episodes.columns]
for metric, _ in JOINT_METRICS:
    if metric not in episodes.columns:
        print(f"note: '{metric}' is not loaded (METRICS / eval CSVs) — its block is left out of the joint table")

# Per-episode metric stats, flattened to "<metric>_<stat>" so the CSV header reads directly.
joint_metric_stats = episodes.groupby(_keys)[[metric for metric, _ in joint_metrics]].agg(list(_JOINT_STATS))
joint_metric_stats.columns = [f"{metric}_{stat}" for metric, stat in joint_metric_stats.columns]

# Outer-joined on (group, eval_cfg, algo, variant): a strip without trajectories keeps its
# eval-CSV stats and shows NaN in the EV / BES columns instead of dropping out.
joint_summary = _in_figure_order(pd.concat(
    [joint_metric_stats,
     ev_success[[column for column, _, _ in JOINT_EV_COLUMNS]],
     bes_summary[[column for column, _, _ in JOINT_BES_COLUMNS]]],
    axis=1,
))

for group_name in group_labels:
    block = joint_summary.loc[group_name]
    for setting in group_setting_labels[group_name]:
        if setting not in block.index.get_level_values("eval_cfg"):
            continue
        table = block.loc[setting]
        rows = []
        for key, row in table.iterrows():
            cells = [cell
                     for metric, _ in joint_metrics
                     for cell in (_latex_cell(row[f"{metric}_mean"], row[f"{metric}_std"]),
                                  _latex_cell(row[f"{metric}_median"]))]
            cells += [_latex_cell(row[column], decimals=decimals) for column, _, decimals in _joint_single_columns]
            rows.append((key, cells))
        body = _latex_body(
            [f"Auto-generated by sps_eval_stats_table_algo_cmp.ipynb — joint summary, "
             f"group '{group_name}', setting '{setting}'.",
             "Metric blocks: mean+-std and median over the eval episodes. 'sessions' = charging sessions "
             "with a disconnect verdict (still connected at the horizon: excluded);",
             f"'met @tol' = of those, SoC at disconnect >= target - {EV_SOC_TOLERANCE}; 'ignored' = of those, "
             f"SoC never moves >= {EV_IGNORED_SOC_CHANGE} away from the arrival SoC while plugged in.",
             "BES columns: per-episode values averaged over the eval episodes (min/max SoC = mean of each "
             "episode's extreme).",
             "Columns: algorithm & variant"
             + "".join(f" & {head} mean+-std & {head} median" for _, head in joint_metrics)
             + "".join(f" & {head}" for _, head, _ in _joint_single_columns) + "."],
            rows, 2 * len(joint_metrics) + len(_joint_single_columns))
        _export(group_name, f"joint summary ({setting})", table, body)
        print(f"\n=== {group_name} / joint summary ({setting}) ===")
        display(table.round(3))

saved base_20260925_133745_out_11.{csv,tex}  (base, joint summary (default))

=== base / joint summary (default) ===


,,total_reward_mean,total_reward_std,total_reward_median,cum_price_EUR_mean,cum_price_EUR_std,cum_price_EUR_median,cum_E_kWh_mean,cum_E_kWh_std,cum_E_kWh_median,sessions,met@tol,ignored,charged_kWh,discharged_kWh,charge_ratio,dir_changes,min_soc,max_soc
algo,variant,,,,,,,,,,,,,,,,,,
PPO,base,-38.688,16.574,-33.769,0.177,3.427,-0.219,12.899,39.879,16.738,18,8,6,9.730,12.748,0.731,3.4,0.1,0.706
SAC,base,-52.841,45.216,-32.641,0.565,1.706,0.274,8.055,17.710,10.294,18,7,5,27.261,16.679,1.687,7.4,0.1,0.950


saved v2_20260925_133745_out_12.{csv,tex}  (v2, joint summary (default))

=== v2 / joint summary (default) ===


,,total_reward_mean,total_reward_std,total_reward_median,cum_price_EUR_mean,cum_price_EUR_std,cum_price_EUR_median,cum_E_kWh_mean,cum_E_kWh_std,cum_E_kWh_median,sessions,met@tol,ignored,charged_kWh,discharged_kWh,charge_ratio,dir_changes,min_soc,max_soc
algo,variant,,,,,,,,,,,,,,,,,,
PPO,v2,-48.29,62.288,-30.509,0.968,1.150,1.006,16.652,24.285,15.642,18,6,2,1.197,4.215,0.208,2.2,0.1,0.303
SAC,v2,-34.20,38.454,-32.109,1.382,3.612,1.634,-3.182,45.742,-3.358,18,8,1,19.854,18.902,1.072,6.1,0.1,0.950
